# N16 · MinHash 近重复去重

**配套 lab**：L06.3 · 数据工程进阶

**目标**：把 MinHash + LSH 从 paper 公式变成可触摸的数。理解：

1. shingle width 对 jaccard 的影响
2. MinHash signature 是 jaccard 的无偏估计
3. num_perm 与估计方差的关系
4. threshold 选择与假阳性 / 假阴性
5. 短文本的特殊处理

**No-GPU 友好**。

## 1. 运行前预测

1. 两个高度相似句子（一个词不同）的 jaccard 大约是 ____
2. shingle width=1 vs width=3 哪个假阳性更多？
3. num_perm=64 vs 256 哪个 minhash_similarity 与真实 jaccard 偏差更小？
4. threshold=0.5 vs 0.85 哪个会删更多样本？
5. "a b" 与 "a c" 用 width=3 时 jaccard 应是 ____

## 2. shingle width 影响

In [ ]:
from mini_infra.data.minhash_dedup import shingles, jaccard, minhash_signature, signature_similarity, find_near_duplicates

a = "a red fox jumps over the small log"
b = "a red fox jumps over a small log"  # 与 a 仅一词不同
c = "the moon shines on a quiet lake"

print("shingle width 对 jaccard 的影响:")
for w in (1, 2, 3, 5):
    j_ab = jaccard(a, b, width=w)
    j_ac = jaccard(a, c, width=w)
    print(f"  width={w}  jaccard(a,b)={j_ab:.3f}  jaccard(a,c)={j_ac:.3f}")

**观察**：
- width=1 → 仅看 token 集合，假阳性高（a,c 都含 'the' 等常见词）
- width=3 → 看 trigram 序列，区分度好
- width 越大 → 越严格但召回率降低

## 3. MinHash 是 jaccard 的无偏估计

In [ ]:
for num_perm in (32, 64, 128, 256):
    sig_a = minhash_signature(a, num_perm=num_perm, width=3)
    sig_b = minhash_signature(b, num_perm=num_perm, width=3)
    j_real = jaccard(a, b, width=3)
    j_estimate = signature_similarity(sig_a, sig_b)
    err = abs(j_real - j_estimate)
    print(f"num_perm={num_perm:3d}  jaccard={j_real:.4f}  minhash={j_estimate:.4f}  err={err:.4f}")

**结论**：MinHash 估计的方差约 1/sqrt(num_perm)。num_perm=128 时偏差 ≤ 0.09，足够生产。

num_perm=32 在生产中会让大量"边缘对"被错分类，必须 ≥ 128。

## 4. threshold 与去重率

In [ ]:
corpus = [
    "a red fox jumps over the small log",
    "a red fox jumps over a small log",  # 1 词差
    "a red fox jumps over the big log",  # 1 词差（不同方向）
    "the moon shines on a quiet lake",
    "distributed training requires careful shard management",
    "distributed training needs careful shard handling",  # 改写
]

for threshold in (0.5, 0.7, 0.85, 0.95):
    res = find_near_duplicates(corpus, threshold=threshold)
    print(f"threshold={threshold}  dedup_pairs={len(res['duplicate_pairs'])}  ratio={res['dedup_ratio']:.3f}")
    for p in res['duplicate_pairs']:
        L = corpus[p['left']][:50]
        R = corpus[p['right']][:50]
        print(f"    pair: '{L}' <=> '{R}'")

**观察**：
- threshold=0.5 几乎把所有同主题对当成重复——假阳性多
- threshold=0.85 只删真正改写的对——业界常用
- threshold=0.95 仅删几乎完全相同的对——严格去重

## 5. 短文本陷阱

In [ ]:
short_corpus = ["a b", "a c", "a d"]
for w in (1, 3):
    print(f"width={w}")
    for t in short_corpus:
        print(f"  shingles('{t}') = {shingles(t, width=w)}")
    res = find_near_duplicates(short_corpus, threshold=0.5)
    print(f"  dedup pairs at threshold=0.5: {len(res['duplicate_pairs'])}")
    print()

**陷阱**：mini_infra 的 shingles 在文本短于 width 时返回完整 text，导致短文本几乎全互相"匹配"。生产实践：短于 shingle_width 的样本跳过 dedup，或单独处理。

## 6. 运行后反思

| 预测项 | 你的预测 | 实际 | ✓/✗ |
|---|---|---|---|
| 一词差 jaccard | | ~0.5-0.8 | |
| width=1 假阳性 | | 多 | |
| num_perm=64 vs 256 | | 256 偏差小 | |
| threshold=0.5 删更多 | | 是 | |

**回到 lab**：
- 跑全集 dedup 前抽样 100 对人工标注真重复率
- 默认配置：shingle_width=3, num_perm=128, threshold=0.85
- 短文本（< 3 词）跳过 dedup
- 出错时去 ticket `dedup_collision_002`